# 05 - Inference Pipeline & Application Readiness (Phase 5)

**Project:** AI-Based Fake News Detection and News Credibility Analysis Using Machine Learning and Evidence Retrieval

**Goal:** take the model trained and saved in Phase 4 and make it *usable*: inspect and verify the saved
artifacts, confirm the final model choice from the existing results, build an inference pipeline
(raw text -> cleaning -> TF-IDF -> model -> readable prediction), test it on new inputs, and prepare it for
a future frontend/backend.

**Nothing is retrained here.** The notebook loads the artifacts saved by `03_ml_models.ipynb` and does not need
the dataset. Label convention (unchanged): `1 = Fake`, `0 = Real`.

## 1. Setup

In [1]:
import json
import re
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.predict import FakeNewsPredictor, format_prediction, UNDETERMINED, DEFAULT_MODELS_DIR
from src.preprocessing.text_cleaning import clean_single_text

MODELS_DIR = DEFAULT_MODELS_DIR
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

print("Project root:", PROJECT_ROOT)
print("scikit-learn in this environment:", sklearn.__version__)

Project root: .../fake-news-detector
scikit-learn in this environment: 1.8.0


## 2. Inspect the saved artifacts

Phase 4 saved three files into `models/`. First: what is there, and what does the metadata say?

In [2]:
rows = []
for f in sorted(MODELS_DIR.iterdir()):
    if f.name.startswith("."):
        continue
    rows.append({"file": f.name, "format": f.suffix, "size_kb": round(f.stat().st_size / 1024, 1)})
print(pd.DataFrame(rows).to_string(index=False))

metadata = json.loads((MODELS_DIR / "model_metadata.json").read_text())
print()
for key in ["model_name", "model_class", "feature_type", "ngram_range", "max_features", "min_df", "max_df",
            "vocabulary_size", "training_dataset", "training_set_size", "selected_using", "label_convention",
            "excluded_features", "validation_metrics"]:
    print(f"{key:20s}: {metadata[key]}")

                   file  format  size_kb
      best_model.joblib .joblib    235.2
    model_metadata.json   .json      1.3
tfidf_vectorizer.joblib .joblib   1150.1

model_name          : Logistic Regression
model_class         : LogisticRegression
feature_type        : TF-IDF
ngram_range         : [1, 2]
max_features        : 30000
min_df              : 3
max_df              : 0.9
vocabulary_size     : 30000
training_dataset    : ISOT Fake and Real News Dataset
training_set_size   : 27050
selected_using      : validation_set
label_convention    : {'1': 'Fake', '0': 'Real'}
excluded_features   : ['subject', 'date', 'article_id']
validation_metrics  : {'accuracy': 0.9876, 'precision': 0.9915, 'recall': 0.9809, 'f1': 0.9862, 'roc_auc': 0.9992}


### Library-version check

Pickled scikit-learn objects are only guaranteed to load correctly under the scikit-learn version that saved
them. We compare the version stored inside each artifact, the version pinned in `requirements.txt`, and the
version running now.

In [3]:
def stored_sklearn_version(path):
    """Read the scikit-learn version string stored inside a joblib/pickle file."""
    match = re.search(rb"_sklearn_version.{0,12}?(\d+\.\d+\.\d+)", path.read_bytes())
    return match.group(1).decode() if match else "unknown"

pinned = re.search(r"scikit-learn==([\d.]+)", (PROJECT_ROOT / "requirements.txt").read_text())
print("Saved with (best_model.joblib)       :", stored_sklearn_version(MODELS_DIR / "best_model.joblib"))
print("Saved with (tfidf_vectorizer.joblib) :", stored_sklearn_version(MODELS_DIR / "tfidf_vectorizer.joblib"))
print("Pinned in requirements.txt           :", pinned.group(1) if pinned else "not pinned")
print("Running now                          :", sklearn.__version__)

Saved with (best_model.joblib)       : 1.8.0
Saved with (tfidf_vectorizer.joblib) : 1.8.0
Pinned in requirements.txt           : 1.5.2
Running now                          : 1.8.0


## 3. Verify the artifacts load and belong together

`FakeNewsPredictor` (in `src/models/predict.py`) loads the model, the vectorizer and the metadata, and
**refuses to start** if they disagree (vocabulary size, n-gram range, model class, label convention). It also
records any scikit-learn version warning raised while loading.

In [4]:
predictor = FakeNewsPredictor()

print("Loaded model      :", type(predictor.model).__name__)
print("Loaded vectorizer :", type(predictor.vectorizer).__name__)
print("Model classes_    :", predictor.model.classes_.tolist(), "(0 = Real, 1 = Fake)")
print("Model features    :", predictor.model.coef_.shape[1])
print("Vocabulary size   :", len(predictor.vectorizer.vocabulary_))
print("Load warnings     :", predictor.load_warnings or "none")
print()
v = predictor.vectorizer
print("Vectorizer settings preserved from training:")
print(f"  ngram_range={v.ngram_range}, min_df={v.min_df}, max_df={v.max_df}, lowercase={v.lowercase}, "
      f"strip_accents={v.strip_accents!r}, sublinear_tf={v.sublinear_tf}, idf fitted={hasattr(v, 'idf_')}")

Loaded model      : LogisticRegression
Loaded vectorizer : TfidfVectorizer
Model classes_    : [0, 1] (0 = Real, 1 = Fake)
Model features    : 30000
Vocabulary size   : 30000
Load warnings     : none

Vectorizer settings preserved from training:
  ngram_range=(1, 2), min_df=3, max_df=0.9, lowercase=True, strip_accents='unicode', sublinear_tf=True, idf fitted=True


### Is this the same model that Phase 4 analysed?

Notebook 03 (Section 12) printed the model's strongest coefficients. If the saved file is the same model, the
coefficients must match those stored printed values.

In [5]:
vocab = predictor.vectorizer.vocabulary_
coef = predictor.model.coef_[0]
expected_from_notebook_03 = {"via": 8.584, "reuters": -16.792, "read more": 4.930, "said": -9.822}

for term, printed in expected_from_notebook_03.items():
    actual = coef[vocab[term]]
    ok = abs(actual - printed) < 0.005
    print(f"{term:10s} notebook 03: {printed:8.3f} | loaded artifact: {actual:8.3f} | match: {ok}")
    assert ok, f"Coefficient mismatch for '{term}'"
print("\nThe saved artifact reproduces the coefficients analysed in notebook 03.")

via        notebook 03:    8.584 | loaded artifact:    8.584 | match: True
reuters    notebook 03:  -16.792 | loaded artifact:  -16.792 | match: True
read more  notebook 03:    4.930 | loaded artifact:    4.930 | match: True
said       notebook 03:   -9.822 | loaded artifact:   -9.822 | match: True

The saved artifact reproduces the coefficients analysed in notebook 03.


## 4. Final model selection (from the existing results)

No models are retrained. The comparison table and the one-time test result are read from the **outputs already
stored** in notebooks 03 and 04, so the numbers here are the actual Phase 4 results, not retyped ones.

In [6]:
def notebook_output_texts(nb_path):
    nb = json.loads(Path(nb_path).read_text(encoding="utf-8"))
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        for out in cell.get("outputs", []):
            text = out.get("text") or out.get("data", {}).get("text/plain") or ""
            yield "".join(text) if isinstance(text, list) else text

# --- validation comparison table (notebook 03, Section 8) ---
table_text = next(t for t in notebook_output_texts(NOTEBOOKS_DIR / "03_ml_models.ipynb")
                  if "Accuracy" in t and "ROC-AUC" in t and "Linear SVM" in t)
row_pattern = re.compile(r"^\d+\s+(\S.*?)\s+(0\.\d+)\s+(0\.\d+)\s+(0\.\d+)\s+(0\.\d+)\s+(0\.\d+)", re.M)
comparison = pd.DataFrame(
    [m.groups() for m in row_pattern.finditer(table_text)],
    columns=["Model", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
).set_index("Model").astype(float)
assert len(comparison) == 5, "Expected 5 models in the stored comparison table"

print("Validation results (positive class = Fake), parsed from notebook 03:")
print(comparison.to_string())

# The saved metadata must agree with this table for the selected model.
lr_row = comparison.loc["Logistic Regression"]
for metric, key in [("Accuracy", "accuracy"), ("Precision", "precision"), ("Recall", "recall"),
                    ("F1", "f1"), ("ROC-AUC", "roc_auc")]:
    assert abs(lr_row[metric] - metadata["validation_metrics"][key]) < 1e-4, metric
print("\nCross-check passed: model_metadata.json matches the Logistic Regression row above.")

Validation results (positive class = Fake), parsed from notebook 03:
                     Accuracy  Precision  Recall      F1  ROC-AUC
Model                                                            
Linear SVM             0.9959     0.9966  0.9943  0.9954   0.9999
Logistic Regression    0.9876     0.9915  0.9809  0.9862   0.9992
XGBoost (SVD)          0.9838     0.9858  0.9782  0.9820   0.9984
Random Forest (SVD)    0.9646     0.9715  0.9496  0.9604   0.9940
Naive Bayes            0.9558     0.9563  0.9454  0.9508   0.9906

Cross-check passed: model_metadata.json matches the Logistic Regression row above.


### What the metrics say, and what they cost

For a fake-news detector the metrics that matter most are **Recall on Fake** (a missed fake article is the
costly error), **Precision on Fake** (real articles wrongly flagged), **F1**, and **ROC-AUC**. Accuracy alone is
less informative because the classes are not perfectly balanced.

On those metrics the table has one clear leader. Below, the recall/precision gap is converted into actual
article counts on the validation set (2,618 Fake and 3,179 Real articles, per notebook 03). The method is
checked against the counts notebook 03 reported for Logistic Regression (22 false positives, 50 false negatives).

In [7]:
N_FAKE_VAL = 2618   # Fake articles in the validation set (support shown in notebook 03's classification report)

def error_counts(model_name):
    recall, precision = comparison.loc[model_name, "Recall"], comparison.loc[model_name, "Precision"]
    true_fake = recall * N_FAKE_VAL
    missed_fake = N_FAKE_VAL - true_fake                    # Fake predicted as Real
    wrongly_flagged = true_fake * (1 / precision - 1)       # Real predicted as Fake
    return round(missed_fake), round(wrongly_flagged)

summary = pd.DataFrame(
    {name: error_counts(name) for name in comparison.index},
    index=["Fake missed (false negatives)", "Real wrongly flagged (false positives)"],
).T
print("Approximate validation errors implied by each model's recall/precision:")
print(summary.to_string())
print()
print("Check against notebook 03 for Logistic Regression: reported 50 false negatives and 22 false positives ->",
      "derived:", summary.loc["Logistic Regression"].tolist())

# --- one-time test-set result for the selected model (notebook 04) ---
test_text = next(t for t in notebook_output_texts(NOTEBOOKS_DIR / "04_model_evaluation.ipynb")
                 if "FINAL TEST SET EVALUATION" in t and "Accuracy" in t)
test_metrics = {name: float(re.search(name + r"[^:]*:\s*([0-9.]+)", test_text).group(1))
                for name in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]}
print("\nLogistic Regression on the held-out TEST set (notebook 04, evaluated once):", test_metrics)

Approximate validation errors implied by each model's recall/precision:
                     Fake missed (false negatives)  Real wrongly flagged (false positives)
Linear SVM                                      15                                       9
Logistic Regression                             50                                      22
XGBoost (SVD)                                   57                                      37
Random Forest (SVD)                            132                                      73
Naive Bayes                                    143                                     113

Check against notebook 03 for Logistic Regression: reported 50 false negatives and 22 false positives -> derived: [50, 22]

Logistic Regression on the held-out TEST set (notebook 04, evaluated once): {'Accuracy': 0.9879, 'Precision': 0.9927, 'Recall': 0.9805, 'F1': 0.9865, 'ROC-AUC': 0.9982}


### Decision: keep Logistic Regression as the final model

**Linear SVM scores higher than Logistic Regression on every validation metric** (see the table). This is
stated plainly because the Phase 4 selection did not pick the top scorer. Logistic Regression is kept as the
final model, and the reasons are specific:

1. **It outputs a real probability** (`predict_proba`). The planned credibility-scoring stage (Phase 8) needs a
   continuous ML score to combine with the LLM and evidence signals. `LinearSVC` only gives an unbounded
   `decision_function` score and would need extra calibration work first.
2. **It is already trained, saved, and evaluated once on the test set** (test F1 0.9865 vs validation 0.9862,
   so performance is consistent across the two held-out splits). Linear SVM was never saved. Switching would
   need the dataset (not in the project ZIP) and a retrain.
3. **It is interpretable**, which is what exposed the dataset-artifact reliance in Phase 4.
4. **Its performance is still strong**: second-best on every metric, and ahead of XGBoost, Random Forest and
   Naive Bayes.

**The trade-off is real:** on validation, Linear SVM would miss roughly a third as many Fake articles (see the
error-count table above). If the project later prioritises raw detection over a probability output, the
alternative is Linear SVM + probability calibration. That would be a deliberate change for you to approve, not
something to do silently.

**Final model: Logistic Regression + TF-IDF** (`models/best_model.joblib` + `models/tfidf_vectorizer.joblib`).

## 5. The inference pipeline

```
raw article text
  -> clean_single_text()                (src/preprocessing/text_cleaning.py, same function Phase 2 used)
  -> TF-IDF vectorizer .transform()     (fitted on the training set only; never refit)
  -> Logistic Regression predict_proba  (P(Fake))
  -> readable prediction dict
```

Phase 4 trained on the cleaned article **body** only (`clean_text`), so inference takes the article body, not the
title. The cell below walks through the steps by hand for one input, then confirms the packaged
`predictor.predict()` gives the identical probability.

In [8]:
sample = ("WASHINGTON (Reuters) - The U.S. Senate on Tuesday passed a spending bill that funds the government "
          "through September, officials said, sending the measure to the House of Representatives, which is "
          "expected to vote on Wednesday.")

clean = clean_single_text(sample)                                 # step 1: cleaning
X = predictor.vectorizer.transform([clean])                       # step 2: TF-IDF (transform only)
p_fake = predictor.model.predict_proba(X)[0, list(predictor.model.classes_).index(1)]   # step 3: P(Fake)

print("Cleaned text      :", clean[:90] + "...")
print("TF-IDF matrix     :", X.shape, f"({X.nnz} non-zero terms)")
print(f"P(Fake) by hand   : {p_fake:.6f}")

packaged = predictor.predict(sample)
print(f"P(Fake) predictor : {packaged['fake_probability']}")
assert abs(p_fake - packaged["fake_probability"]) < 1e-4
print("\nManual steps and the packaged predictor agree.")

Cleaned text      : WASHINGTON (Reuters) - The U.S. Senate on Tuesday passed a spending bill that funds the go...
TF-IDF matrix     : (1, 30000) (44 non-zero terms)
P(Fake) by hand   : 0.000617
P(Fake) predictor : 0.0006

Manual steps and the packaged predictor agree.


## 6. Test on new inputs

**These test articles were written by hand for this notebook. They are not from the dataset, and the events
described are invented.** They test whether the pipeline works and behaves sensibly. They are *not* a measure
of accuracy (the accuracy figures are the validation/test results above).

Cases A-D are typical examples of each class's style. Cases E-F are deliberately adversarial: they swap the
*style* and the *truthfulness* around, to check whether the model is reading content or source formatting
(the concern raised in Phase 3/4). The "intended" column is what I designed each case to be, and all results
are reported as they came out, with no re-writing of cases afterwards.

In [9]:
TEST_CASES = [
    {"id": "A", "intended": "Real", "description": "Wire-service style news report",
     "text": ("LONDON (Reuters) - Britain's economy grew by 0.3 percent in the third quarter, the Office for "
              "National Statistics said on Friday, slightly above the 0.2 percent forecast by economists in a "
              "Reuters poll. Services output rose while manufacturing was flat, the statistics agency said. "
              "Finance minister Jane Holloway said the figures showed the economy was on a steady footing, "
              "but opposition lawmakers argued that households were still struggling with higher prices. "
              "The Bank of England is due to announce its next interest rate decision on Thursday, and "
              "analysts said the data was unlikely to change expectations that rates would be held.")},
    {"id": "B", "intended": "Real", "description": "Local news, plain style (no wire formatting)",
     "text": ("The city council voted 7-2 on Monday night to approve a plan to add protected bike lanes along "
              "Maple Avenue, a project that has been debated for more than a year. Supporters told the "
              "council that the lanes would make the busy road safer for cyclists and pedestrians, citing "
              "three collisions on the street last year. Some business owners said they feared losing "
              "parking spaces. The council agreed to review the effect on local shops after six months. "
              "Construction is expected to begin in the spring and take about eight weeks, according to the "
              "city's transportation department.")},
    {"id": "C", "intended": "Fake", "description": "Clickbait style with image credit and 'read more'",
     "text": ("You will NOT believe what they are hiding from you about this one simple trick! Politicians "
              "are terrified that you will find out the truth, and the mainstream media refuses to cover "
              "it. Watch the video below and share it with everyone you know before it gets deleted. "
              "Featured image via Getty Images. Read more: click here to see the shocking details. "
              "Sign up for our newsletter and follow us on Facebook for more stories they do not want "
              "you to see.")},
    {"id": "D", "intended": "Fake", "description": "Hyper-partisan rant style",
     "text": ("BREAKING: Obama and the radical left just got CAUGHT again and the GOP is demanding answers! "
              "Can you believe these people? They think we are all stupid, but America is waking up. "
              "President Trump has been telling us for years that the system is rigged, and now even the "
              "liberals are admitting it. This is a disgrace and every American should be outraged. "
              "Watch this video and tell us what you think in the comments below. Image via Twitter.")},
    {"id": "E", "intended": "Fake", "description": "ADVERSARIAL: invented false claim written in wire-service style",
     "text": ("WASHINGTON (Reuters) - The federal government will ban all cash transactions starting this "
              "Friday and require every citizen to use a government-issued digital wallet, a senior "
              "administration official said on Tuesday. The official, who spoke on condition of "
              "anonymity, said the measure would take effect nationwide without a vote in Congress. "
              "Banks were told on Monday to freeze existing paper currency deposits, the official said. "
              "The White House did not respond to a request for comment on Tuesday.")},
    {"id": "F", "intended": "Real", "description": "ADVERSARIAL: neutral factual report with an image credit",
     "text": ("The national weather service said on Sunday that a cold front will bring rain to most of the "
              "region early this week, with temperatures expected to drop by about ten degrees by Wednesday. "
              "Forecasters advised drivers to allow extra time on the roads during the morning commute. "
              "Rainfall totals are expected to reach between one and two inches in some areas. The rain is "
              "forecast to clear by Thursday afternoon, followed by sunny weather over the weekend. "
              "Image via Getty Images. Read more about the forecast on the weather service website.")},
    {"id": "G", "intended": "n/a", "description": "Non-English text (Hindi)",
     "text": "प्रधानमंत्री ने आज नई आर्थिक नीति की घोषणा की और कहा कि इससे देश के किसानों को लाभ होगा।"},
    {"id": "H", "intended": "n/a", "description": "Headline-length text only",
     "text": "Reuters said on Tuesday the bill passed."},
]

results = predictor.predict_batch([c["text"] for c in TEST_CASES], explain=True, top_n=3)

table = pd.DataFrame([{
    "case": c["id"], "intended": c["intended"], "predicted": r["label"],
    "P(Fake)": r["fake_probability"], "confidence": r["confidence_level"],
    "words": r["n_words"], "known_terms": r["n_known_terms"],
} for c, r in zip(TEST_CASES, results)])
print(table.to_string(index=False))

case intended    predicted  P(Fake) confidence  words  known_terms
   A     Real         Real   0.0032       high    101          101
   B     Real         Real   0.2888   moderate     97          100
   C     Fake         Fake   0.9994       high     82          103
   D     Fake         Fake   0.9995       high     79          103
   E     Fake         Real   0.0019       high     78           92
   F     Real         Fake   0.6986        low     91           91
   G      n/a Undetermined      NaN        NaN     19            0
   H      n/a         Real   0.0019       high      7           10


In [10]:
for case, r in zip(TEST_CASES, results):
    print(f"--- Case {case['id']}: {case['description']} (intended: {case['intended']})")
    print(format_prediction(r).split("Note")[0].rstrip())
    print()

--- Case A: Wire-service style news report (intended: Real)
Prediction : Real  (confidence 99.7% - high)
P(Fake)    : 0.3%    P(Real): 99.7%
Text size  : 101 words, 101 known model terms
Toward Fake: is (+0.07), that (+0.06), while (+0.04)
Toward Real: reuters (-0.95), said (-0.62), said on (-0.34)

--- Case B: Local news, plain style (no wire formatting) (intended: Real)
Prediction : Real  (confidence 71.1% - moderate)
P(Fake)    : 28.9%    P(Real): 71.1%
Text size  : 97 words, 100 known model terms
Toward Fake: that the (+0.08), according (+0.08), according to (+0.08)
Toward Real: said (-0.26), on monday (-0.24), year (-0.16)

--- Case C: Clickbait style with image credit and 'read more' (intended: Fake)
Prediction : Fake  (confidence >99.9% - high)
P(Fake)    : >99.9%    P(Real): <0.1%
Text size  : 82 words, 103 known model terms
Toward Fake: read more (+0.46), you (+0.46), via (+0.44)
Toward Real: not want (-0.03), click (-0.03), do not (-0.03)

--- Case D: Hyper-partisan rant styl

In [11]:
scored = [(c, r) for c, r in zip(TEST_CASES, results) if c["intended"] in ("Real", "Fake")]
typical = [(c, r) for c, r in scored if c["id"] in "ABCD"]
adversarial = [(c, r) for c, r in scored if c["id"] in "EF"]

def n_match(pairs):
    return sum(r["label"] == c["intended"] for c, r in pairs)

print(f"Typical-style cases (A-D)  : prediction matched my intended label in {n_match(typical)} of {len(typical)}")
print(f"Adversarial cases (E-F)    : prediction matched my intended label in {n_match(adversarial)} of {len(adversarial)}")
print(f"Pipeline errors across all {len(TEST_CASES)} inputs: 0 (every input returned a result)")

Typical-style cases (A-D)  : prediction matched my intended label in 4 of 4
Adversarial cases (E-F)    : prediction matched my intended label in 0 of 2
Pipeline errors across all 8 inputs: 0 (every input returned a result)


## 7. Edge cases and bad input

In [12]:
bad_inputs = {"empty string": "", "whitespace only": "   \n\t  ", "None": None, "a number": 42}
for name, value in bad_inputs.items():
    try:
        predictor.predict(value)
        print(f"{name:16s}: NO ERROR RAISED (unexpected)")
    except ValueError as err:
        print(f"{name:16s}: ValueError -> {err}")

print()
messy = "<p>WASHINGTON (Reuters) -   The senator said\u00a0on Tuesday that the \u201cbill\u201d was ready &amp; talks would continue.</p>"
r = predictor.predict(messy)
print("HTML / curly quotes / extra whitespace ->", r["label"], r["fake_probability"], "(cleaned before TF-IDF)")

nonsense = predictor.predict("zzqx qwxz vvkk xxjq")
print("No known vocabulary ->", nonsense["label"], "|", nonsense["warnings"][-1])

long_text = " ".join([TEST_CASES[0]["text"]] * 200)
r = predictor.predict(long_text)
print(f"Very long input ({r['n_words']:,} words) ->", r["label"], r["fake_probability"])

empty string    : ValueError -> Input #0 is empty after cleaning - please provide article text.
whitespace only : ValueError -> Input #0 is empty after cleaning - please provide article text.
None            : ValueError -> Input #0 must be a string, got NoneType.
a number        : ValueError -> Input #0 must be a string, got int.

HTML / curly quotes / extra whitespace -> Real 0.0009 (cleaned before TF-IDF)
No known vocabulary -> Undetermined | None of the words in this text are in the model's vocabulary (e.g. non-English text), so no prediction can be made.
Very long input (20,200 words) -> Real 0.0063


## 8. Automated tests

`tests/test_inference.py` runs 13 checks against the real saved artifacts (loading, artifact consistency,
inference-vs-training preprocessing equivalence, batch vs single, error handling, edge cases). They are run here
in a subprocess so the result is recorded in the notebook.

In [13]:
proc = subprocess.run([sys.executable, "-m", "unittest", "tests.test_inference", "-v"],
                      cwd=PROJECT_ROOT, capture_output=True, text=True)
lines = (proc.stderr or proc.stdout).strip().splitlines()
print("\n".join(lines[-4:]))
assert proc.returncode == 0, "Inference tests failed"

----------------------------------------------------------------------
Ran 13 tests in 0.423s

OK


## 9. Using the model in an application

**The interface a frontend/backend needs** is `FakeNewsPredictor`. Load it once at start-up, then call
`predict(text)`. The result is a plain dict that is JSON-serialisable, so it can be returned from an API as-is.
The cell below simulates what an API endpoint would do with a JSON request body, including a bad request.

In [14]:
predictor_for_app = FakeNewsPredictor()          # load ONCE when the app starts, reuse for every request

def analyze_request(payload: dict) -> dict:
    """What a POST /predict endpoint would do with a JSON body like {"text": "..."}."""
    try:
        return {"ok": True, "result": predictor_for_app.predict(payload["text"])}
    except KeyError:
        return {"ok": False, "error": "Request body must contain a 'text' field."}
    except ValueError as err:
        return {"ok": False, "error": str(err)}

ok_response = analyze_request({"text": TEST_CASES[0]["text"]})
bad_response = analyze_request({"text": "   "})
missing_field = analyze_request({"title": "no body"})

as_json = json.dumps(ok_response)                # proves the whole response survives JSON serialisation
print("Valid request    ->", ok_response["ok"], ok_response["result"]["label"], ok_response["result"]["fake_probability"])
print("Blank text       ->", bad_response)
print("Missing field    ->", missing_field)
print("JSON response size:", len(as_json), "characters")

Valid request    -> True Real 0.0032
Blank text       -> {'ok': False, 'error': 'Input #0 is empty after cleaning - please provide article text.'}
Missing field    -> {'ok': False, 'error': "Request body must contain a 'text' field."}
JSON response size: 505 characters


**Planned FastAPI wiring (Phase 9) - illustrative only, NOT executed in Phase 5.** The existing
`app/backend/main.py` still contains only the Phase 1 `/health` endpoint. FastAPI is not installed in the
environment where Phase 5 was verified, so the endpoint below was deliberately not added to the project
rather than shipped untested. It shows how little glue is needed:

```python
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from src.models.predict import load_predictor

app = FastAPI(title="Fake News Detection API")
predictor = load_predictor()                  # loaded once at start-up

class ArticleRequest(BaseModel):
    text: str

@app.post("/predict")
def predict(request: ArticleRequest):
    try:
        return predictor.predict(request.text, explain=True)
    except ValueError as err:
        raise HTTPException(status_code=422, detail=str(err))
```

Command-line use (verified): `python -m src.models.predict --file article.txt --explain`

## 10. Findings from testing, limitations, and what remains

### What the tests showed

**The pipeline works.** The artifacts load, all 8 test inputs returned a result, invalid input raises clear
errors, and all 13 automated tests pass. Inference-time preprocessing is verified to give the same probabilities as the
Phase 2 -> Phase 4 training path.

**Typical-style articles (A-D): all 4 matched my intended label.** Case B (plain local news with no wire
formatting) was classed Real but with only moderate confidence (P(Fake) = 0.29), where the wire-style Case A was
0.003.

**Adversarial cases (E-F): both went the "wrong" way, and the explanation shows why.**

- **Case E** is an *invented false claim* written in Reuters style. It was classed **Real with high confidence
  (99.8%)**. Its strongest contributing terms were `reuters`, `said`, `on tuesday`.
- **Case F** is a *neutral, factual weather report* with an image credit and a "Read more" line. It was classed
  **Fake (69.9%, low confidence)**, driven by `read more`, `via`, `read`.

This is the Phase 3/4 concern showing up directly in inference: the model is largely reading **source
formatting**, not truthfulness. Two hand-written cases are an illustration, not a statistical measurement, but they
are consistent with the coefficient analysis in notebook 03 and should be mentioned in the project report's
limitations.

**Confidence is not reliability.** Case E was wrong at "high" confidence. The probabilities have not been
calibration-tested, and the High/Moderate/Low labels are display bands only. A very short input (Case H, 7 words)
was also classed Real at 99.8%, almost entirely from `reuters` and `said on`. The pipeline attaches a
short-input warning, but it does not suppress the prediction.

**Non-English text** (Case G) has no overlap with the vocabulary, so the predictor returns `Undetermined` instead
of a meaningless score from the model's intercept.

### Remaining work

- **Phase 7 - evidence retrieval (RAG)**: the ML score should be one input to a hybrid system, given the
  limitations above. (The planned Llama integration was later dropped.)
- **Phase 8 - credibility scoring**: will use `fake_probability` from this predictor.
- **Phase 9 - web application**: wire `FakeNewsPredictor` into `app/backend/main.py` and `app/frontend/app.py`.
- **Decision for you:** keep Logistic Regression (current), or retrain Linear SVM (+ calibration) for stronger raw
  detection (Section 4).
- **Environment:** `requirements.txt` pins scikit-learn 1.5.2, but the artifacts were saved with 1.8.0 (Section 2).
- The original Phase 5 plan in the README (a transformer model) is not part of this phase; it is now unscheduled.

In [15]:
print("PHASE 5 VERIFICATION SUMMARY")
print("=" * 50)
print("Artifacts loaded and cross-checked        : yes")
print("Final model                                :", predictor.metadata["model_name"])
print("Inputs tested                              :", len(TEST_CASES), "hand-written +", len(bad_inputs), "invalid")
print("Unit tests (tests/test_inference.py)       : passed")
print("Retraining performed                       : no")

PHASE 5 VERIFICATION SUMMARY
Artifacts loaded and cross-checked        : yes
Final model                                : Logistic Regression
Inputs tested                              : 8 hand-written + 4 invalid
Unit tests (tests/test_inference.py)       : passed
Retraining performed                       : no
